In [ ]:
# ESTA expressive VO - IndexTTS2 (+ Seed-VC for guide takes)
import subprocess, os, json, glob, shutil, traceback
OUT = "/kaggle/working"
TMP = "/kaggle/temp" if os.path.isdir("/kaggle/temp") else "/tmp/esta"
os.makedirs(TMP, exist_ok=True)
os.environ["HF_HOME"] = TMP + "/hf"
os.environ["GIT_LFS_SKIP_SMUDGE"] = "1"

INP = None
for root, _, files in os.walk("/kaggle/input"):
    if "voice_script.json" in files:
        INP = root
JOB = json.load(open(INP + "/voice_script.json"))
REF = INP + "/ref.wav"
LINES = [l for l in JOB["lines"] if l["id"] in set(["L06", "L07"])] if ["L06", "L07"] else JOB["lines"]
print("[esta] lines to synthesise:", len(LINES), flush=True)
status = {}

def sh(cmd, cwd=None, log="run.log"):
    print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    with open(OUT + "/" + log, "a") as fh:
        fh.write("$ " + cmd + "\n" + r.stdout + r.stderr + "\n")
    print((r.stdout + r.stderr)[-2000:], flush=True)
    if r.returncode:
        raise RuntimeError(cmd + " -> exit " + str(r.returncode))

sh("pip install -q uv")

# ---- IndexTTS2: every line without a guide take ------------------------------
TTS_LINES = [l for l in LINES if l["engine"] == "indextts2"]
IDX = TMP + "/index-tts"
IDX_SCRIPT = r"""
import json, sys, traceback
from indextts.infer_v2 import IndexTTS2
cfg = json.load(open(sys.argv[1]))
tts = IndexTTS2(cfg_path="checkpoints/config.yaml", model_dir="checkpoints",
                use_fp16=True, use_cuda_kernel=False, use_deepspeed=False)
done = {}
for line in cfg["lines"]:
    kw = {}
    if line.get("emo_vector"):
        # Explicit vector — skips the text classifier entirely.
        kw = dict(emo_vector=line["emo_vector"])
    elif line.get("emotion"):
        kw = dict(use_emo_text=True, emo_text=line["emotion"],
                  emo_alpha=line.get("emo_alpha") or 0.65)
    # A line may name its own speaker ({voice: ...}) — a quoted passage, a
    # second character. Everything else clones the session's ref.
    spk = cfg["inp"] + "/" + line["voice_file"] if line.get("voice_file") else cfg["ref"]
    try:
        tts.infer(spk_audio_prompt=spk, text=line["model_text"],
                  output_path=cfg["out"] + "/line_" + line["id"] + ".wav",
                  use_random=False, verbose=False, **kw)
        done[line["id"]] = "ok"
    except Exception:
        done[line["id"]] = traceback.format_exc()[-500:]
    print(line["id"], done[line["id"]][:120], flush=True)
json.dump(done, open(cfg["out"] + "/idx_lines.json", "w"), indent=1)
"""
if TTS_LINES:
    try:
        sh("git clone --depth 1 https://github.com/index-tts/index-tts " + IDX)
        sh("uv sync", cwd=IDX)
        sh("uv run python -c \"from huggingface_hub import snapshot_download; "
           "snapshot_download('IndexTeam/IndexTTS-2', local_dir='checkpoints')\"", cwd=IDX)
        open(IDX + "/esta_cfg.json", "w").write(json.dumps(
            {"lines": TTS_LINES, "ref": REF, "inp": INP, "out": OUT}))
        open(IDX + "/esta_run.py", "w").write(IDX_SCRIPT)
        sh("uv run python esta_run.py esta_cfg.json", cwd=IDX)
        status["indextts2"] = "ok"
    except Exception:
        status["indextts2"] = traceback.format_exc()[-1500:]

# ---- Seed-VC: lines the user acted out themselves -----------------------------
VC_LINES = [l for l in LINES if l["engine"] == "seedvc"]
SVC = TMP + "/seed-vc"
if VC_LINES:
    try:
        sh("git clone --depth 1 https://github.com/Plachtaa/seed-vc " + SVC)
        # plain pip, not uv: seed-vc's requirements.txt has a quoted `--pre` line uv can't parse
        sh("uv venv --python 3.10 --seed .venv", cwd=SVC)
        sh(".venv/bin/python -m pip install -q -r requirements.txt", cwd=SVC)
        done = {}
        for line in VC_LINES:
            src = INP + "/" + line.get("guide_file", "guide__" + os.path.basename(line["guide"]))
            try:
                sh(".venv/bin/python inference.py --source " + src + " --target " + REF +
                   " --output " + TMP + "/vc/" + line["id"] +
                   " --diffusion-steps 30 --length-adjust 1.0 --inference-cfg-rate 0.7"
                   " --f0-condition False --auto-f0-adjust False --semi-tone-shift 0"
                   " --fp16 True", cwd=SVC)
                got = sorted(glob.glob(TMP + "/vc/" + line["id"] + "/*.wav"))
                shutil.copy(got[-1], OUT + "/line_" + line["id"] + ".wav")
                done[line["id"]] = "ok"
            except Exception:
                done[line["id"]] = traceback.format_exc()[-500:]
        json.dump(done, open(OUT + "/vc_lines.json", "w"), indent=1)
        status["seedvc"] = "ok"
    except Exception:
        status["seedvc"] = traceback.format_exc()[-1500:]

status["produced"] = sorted(os.path.basename(p) for p in glob.glob(OUT + "/line_*.wav"))
json.dump(status, open(OUT + "/status.json", "w"), indent=1)
print("ESTA_VO_STATUS::" + json.dumps({k: v for k, v in status.items() if k != "produced"}), flush=True)
